# Comparison of 2 calibrations 

Comparision of two method of calibration:

- Brucker software 
- Automatic inhouse calibraiton base on mass difference map error

In [ ]:
import spectre_analysis as sp_ana
import calibration as cal
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import os
import gc

from pathlib import Path
from scipy.stats import gaussian_kde
from typing import Dict, Union, Optional, Tuple, List

#import datashader as ds
#from datashader import transfer_functions as tf
#import colorcet as cc

## Data extraction

In [ ]:
workers = sp_ana.find_workers()

path_std = Path.cwd().parent / "data" / "standards"
path_cal = Path.cwd().parent / "data" / "Standard_Gums_Calibrated_files"

list_std = os.listdir(path_std)
list_cal = os.listdir(path_cal)
list_cal = [i.replace("_Calibrated", "") for i in list_cal]

# take the commun spectra of the both folders
spectra = sorted(list(set(list_std) & set(list_cal)))

In [ ]:
# 1) All monomers

monomers_list = ["dHex", "Hex", "HexA", "HexN", "HexNac", "Pent", "Neu", "NeuAc", "NeuGc"]
library = Path.cwd().parent / "data" / "oligo_combinations_clean.csv"
oligo_ref_all = sp_ana.adjust_library(library, monomers_list)
# Transform the mass in ppm and int to optimize the algorithm
for adduct in ['labeled_mass_H+', 'labeled_mass_Na+', 'labeled_mass_K+']:
    oligo_ref_all = sp_ana.normalize_mz_column(
    lf = oligo_ref_all,
    mz_col=adduct,
    new_col=f"{adduct}_int"
)

# 2) Selection of mmonomers

monomers_list = ["Hex", "Pent"]
library = Path.cwd().parent / "data" / "oligo_combinations.csv"
oligo_ref_selection = sp_ana.adjust_library(library, monomers_list)
# Transform the mass in ppm and int to optimize the algorithm
for adduct in ['labeled_mass_H+', 'labeled_mass_Na+', 'labeled_mass_K+']:
    oligo_ref_selection = sp_ana.normalize_mz_column(
    lf = oligo_ref_selection,
    mz_col=adduct,
    new_col=f"{adduct}_int"
)

## Calibration

In [ ]:
def save_spectrum(spectrum, path, spectrum_name ):
    file_name = Path(path) / spectrum_name
    print(file_name)
    spectrum.collect().write_csv(file_name, separator=',')

In [ ]:
path_spectrum_all = Path.cwd().parent / "data" / "calibration" / "spectrum" / "all"
path_spectrum_selection = Path.cwd().parent / "data" / "calibration" / "spectrum" / "selection"
path_peaks_all = Path.cwd().parent / "data" / "calibration" / "peaks" / "all"
path_peaks_selection = Path.cwd().parent / "data" / "calibration" / "peaks" / "selection"
path_results = Path.cwd().parent / "data" / "calibration" / "results"

In [ ]:
for spectrum_name in spectra:

    spectrum_file = Path.cwd().parent / "data" / "Standard_Gums_Calibrated_files" / (
    spectrum_name.replace(".csv", "") + "_Calibrated.csv")
    spectrum_bruker = sp_ana.read_spectrum(spectrum_file)

    # Spectrum extraction with progress logging
    print(f"[INFO] Starting analysis for {spectrum_name}")
    spectrum_file = Path.cwd().parent / "data" / "standards" / spectrum_name
    spectrum_std: pl.LazyFrame = (
        sp_ana.read_spectrum(spectrum_file)
        .pipe(lambda df: df.lazy())  # Ensure LazyFrame if not already
    )
    
    # Peak detection pipeline
    print("[INFO] Extracting peaks with adaptive parameters")
    peaks: pl.LazyFrame = (
        sp_ana.extract_and_detect_peaks(
            spectrum_data=spectrum_std,
            window=500,
            dist_min=0,
            prominence=3e5,
            n_workers=workers
        )
    )

    # Mass normalization
    print("[INFO] Normalizing m/z values")
    peaks = sp_ana.normalize_mz_column(
        lf=peaks,
        mz_col="m/z",
        new_col="mz_int"
    )

    # 1. All
    print("[INFO] Calibration all monomers")
    corrected_spectrum_all = cal.process_calibration(
        spectrum_name,
        spectrum_std,
        peaks,
        oligo_ref_all,
        path_spectrum_all,
        "all_mono"
    )

    # 2. Selection
    print("[INFO] Calibration selection monomers")
    corrected_spectrum_selection = cal.process_calibration(
        spectrum_name,
        spectrum_std,
        peaks,
        oligo_ref_selection,
        path_spectrum_selection,
        "selected_mono"
    )

     # Detect peak on new spectrum
    print("[INFO] Detected new peaks")
    peaks_all = sp_ana.extract_and_detect_peaks(
        spectrum_data=corrected_spectrum_all,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=1,
        mz_col="all_mono",
        intensity_col="Intensity"
    ).collect().lazy()

    peaks_selected = sp_ana.extract_and_detect_peaks(
        spectrum_data=corrected_spectrum_selection,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=1,
        mz_col="selected_mono",
        intensity_col="Intensity"
    ).collect().lazy()

    peaks_bruker = sp_ana.extract_and_detect_peaks(
        spectrum_data=spectrum_bruker,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=1,
    ).rename({"m/z":"bruker"}).collect().lazy()

    # Join
    print('[INFO] Join the spectrum')
    peaks_idx = peaks.with_row_index("idx")
    peaks_all_idx = peaks_all.with_row_index("idx")
    peaks_selected_idx = peaks_selected.with_row_index("idx")
    peaks_bruker_idx = peaks_bruker.with_row_index("idx")

    global_data = (
        peaks_idx
        .join(peaks_all_idx, on="idx", how="inner", suffix="_all")
        .join(peaks_selected_idx, on="idx", how="inner", suffix="_sel")
        .join(peaks_bruker_idx, on="idx", how="inner", suffix="_bruker")
        .drop(["Intensity_all", "Intensity_sel", "Intensity_bruker"])
        .with_columns(pl.lit(spectrum_name).alias("Spectrum_Name"))
    )

    save_spectrum(global_data, path_results, spectrum_name)
    # Free memory
    del peaks, peaks_all, peaks_bruker, peaks_selected, corrected_spectrum_all, corrected_spectrum_selection
    del global_data
    gc.collect()





    # Detect peak on new spectrum
    print("Detected new peaks")
    peaks_all = sp_ana.extract_and_detect_peaks(
        spectrum_data=corrected_spectrum_all,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=1,
        mz_col="all_mono",
        intensity_col="Intensity"
    ).collect()

    peaks_selected = sp_ana.extract_and_detect_peaks(
        spectrum_data=corrected_spectrum_selection,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=1,
        mz_col="selected_mono",
        intensity_col="Intensity"
    ).collect()

    peaks_bruker = sp_ana.extract_and_detect_peaks(
        spectrum_data=spectrum_bruker,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=1,
    ).rename({"m/z":"bruker"}).collect()

    all_spectrum[spectrum_name]=[peaks, peaks_all, peaks_selected, peaks_bruker]

    # Join
    print('Join the spectrum')
    peaks_idx = peaks.with_row_index("idx")
    peaks_all_idx = peaks_all.with_row_index("idx")
    peaks_selected_idx = peaks_selected.with_row_index("idx")
    peaks_bruker_idx = peaks_bruker.with_row_index("idx")

    global_data = (
        peaks_idx
        .join(peaks_all_idx, on="idx", how="inner", suffix="_all")
        .join(peaks_selected_idx, on="idx", how="inner", suffix="_sel")
        .join(peaks_bruker_idx, on="idx", how="inner", suffix="_bruker")
        .drop(["Intensity_all", "Intensity_sel", "Intensity_bruker"])
        .with_columns(pl.lit(spectrum_name).alias("Spectrum_Name"))
    )
    
    
    
    all_spectrum.append(global_data.collect())

    # Free memory
    del peaks, mdim_all_mono, mdim_selection_mono, error_table_all, error_table_selection
    del peaks_all, peaks_bruker, peaks_selected, corrected_spectrum_all, corrected_spectrum_selection
    del global_data
    gc.collect()

print("final concatenation")
final_global_data = pl.concat(all_spectrum,  rechunk=True)
